<a href="https://colab.research.google.com/github/Innovatewithapple/RNNProjects/blob/main/StockPrice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance

In [14]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Dropout,LSTM,Input
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
data = yf.download('AAPL',start='2019-01-01',end='2024-01-01')
data

/tmp/ipykernel_1196/3275127332.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('AAPL',start='2019-01-01',end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2019-01-02,37.503731,37.724594,36.627408,36.784150,148158800
2019-01-03,33.768078,34.606402,33.722955,34.193175,365248800
2019-01-04,35.209625,35.278498,34.150441,34.323805,234428400
2019-01-07,35.131252,35.344992,34.649157,35.314117,219111200
2019-01-08,35.800964,36.055076,35.271372,35.518356,164101200
...,...,...,...,...,...
2023-12-22,191.609467,193.400854,190.985939,193.173208,37149600
2023-12-26,191.065125,191.896484,190.847385,191.619364,28919300


In [4]:
prices = data[['Close']]

In [7]:
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(prices)

In [8]:
len(scaled_data)

1258

In [9]:
# We want the model to look at the last 60 days to predict day 61
window_size = 60
x,y = [],[]

for i in range(window_size,len(scaled_data)):
  x.append(scaled_data[i-window_size:i,0]) # Here it means by slicing we 0 to 59 days of price (0 mean price column), so for ex= if i is 60 that menas 60-60 = [0:60,0] -> array slicing
  y.append(scaled_data[i,0])

x, y = np.array(x), np.array(y)
x.shape

(1198, 60)

In [10]:
# Reshape for LSTM: [Samples, Time_Steps, Features]
x = np.reshape(x, (x.shape[0], x.shape[1], 1))

In [12]:
model = Sequential([
    Input(shape=(window_size,1)), # (Time_Steps, Features)
    LSTM(64,return_sequences=True),
    Dropout(0.2),
    LSTM(64,return_sequences=False),
    Dropout(0.2),
    Dense(25,activation='relu'),
    Dense(1)
])

In [13]:
model.compile(optimizer='adam',loss='mean_squared_error')

In [15]:
callback = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [16]:
model.fit(x,y,epochs=30,batch_size=32,validation_split=0.1,verbose=1,callbacks=[callback],shuffle=False)

Epoch 1/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - loss: 0.0107 - val_loss: 0.0019
Epoch 2/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - loss: 0.0172 - val_loss: 0.0041
Epoch 3/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - loss: 0.0061 - val_loss: 0.0027
Epoch 4/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - loss: 0.0089 - val_loss: 0.0017
Epoch 5/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - loss: 0.0112 - val_loss: 0.0027
Epoch 6/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - loss: 0.0103 - val_loss: 0.0049
Epoch 7/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - loss: 0.0035 - val_loss: 0.0011
Epoch 8/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - loss: 0.0041 - val_loss: 0.0012
Epoch 9/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - loss: 0.0057 - val_loss: 0.0023
Epoch 10/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 0.0032 - val_loss: 0.0010
Epoch 11/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - loss: 0.0029 - val_loss: 0.0010
Epoch 12/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - loss: 0.0